# 04 External Panel Validation

**Scope.** Check aggregated and anonymized external sensory-panel validation inputs together with graph-model comparison summaries, without treating missing results as evidence.

## Expected input files

- `data/external_panel/external_molecules_87.csv`
- `data/external_panel/structural_neighbor_pairs_60.csv`
- `data/external_panel/panel_label_vocabulary_43.csv`
- `data/external_panel/aggregated_panel_ratings.csv`
- `data/external_panel/panel_derived_topk_labels.csv`
- `data/external_panel/pairwise_panel_jaccard_similarity.csv`
- `results/external_validation/graph_pairwise_similarity_predictions.csv`
- `results/external_validation/concordance_coefficients.csv`
- `results/external_validation/embedding_knn_auc.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt

SAVE_OUTPUTS = False


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_configs = (candidate / "configs").is_dir()
        has_readme = (candidate / "README.md").exists()
        if has_configs and has_readme:
            return candidate
        if has_configs:
            fallback = candidate
            return fallback
    raise FileNotFoundError("Could not locate project root containing configs/.")


def load_yaml(path: Path):
    with path.open("r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def check_files(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({"path": path.as_posix(), "exists": path.exists()})
    return pd.DataFrame(rows)


def require_files(paths):
    missing = [Path(path) for path in paths if not Path(path).exists()]
    if missing:
        formatted = "
".join(f"- {path.as_posix()}" for path in missing)
        raise FileNotFoundError(
            "The following required files are missing:
"
            f"{formatted}
"
            "Data required from user later before this notebook can run end-to-end."
        )


def print_missing_file_report(report_df: pd.DataFrame) -> None:
    missing = report_df.loc[~report_df["exists"], "path"].tolist()
    if missing:
        print("Data required from user later:")
        for item in missing:
            print(f"- {item}")
    else:
        print("All expected input files are present.")


PROJECT_ROOT = find_project_root()
CONFIG_DIR = PROJECT_ROOT / "configs"
DATASET_CFG = load_yaml(CONFIG_DIR / "dataset.yaml")


## 1. Scope

This notebook checks the external structural-neighborhood sensory-panel validation data and related prediction summaries. Sensory-panel records are expected to be aggregated and anonymized before public use in this notebook.

## 2. Expected inputs

The notebook expects external molecule metadata, pair definitions, panel vocabulary, aggregated panel summaries, and released external-validation result tables.

## 3. Load configs

In [ ]:
graph_cfg = load_yaml(CONFIG_DIR / 'graph_model_settings.yaml')
expected_files = [
    PROJECT_ROOT / DATASET_CFG['files']['external_molecules'],
    PROJECT_ROOT / DATASET_CFG['files']['structural_neighbor_pairs'],
    PROJECT_ROOT / DATASET_CFG['files']['panel_label_vocabulary'],
    PROJECT_ROOT / DATASET_CFG['files']['aggregated_panel_ratings'],
    PROJECT_ROOT / DATASET_CFG['files']['panel_derived_topk_labels'],
    PROJECT_ROOT / DATASET_CFG['files']['pairwise_panel_jaccard_similarity'],
    PROJECT_ROOT / 'results/external_validation/graph_pairwise_similarity_predictions.csv',
    PROJECT_ROOT / 'results/external_validation/concordance_coefficients.csv',
    PROJECT_ROOT / 'results/external_validation/embedding_knn_auc.csv',
]
print('External set size declared in config:', DATASET_CFG['dataset']['external_n_molecules'])
print('External pair count declared in config:', DATASET_CFG['dataset']['external_n_pairs'])

## 4. File existence checks

In [ ]:
file_report = check_files(expected_files)
display(file_report)
print_missing_file_report(file_report)

## 5. External molecule and pair count checks

In [ ]:
loaded = {}
for path in expected_files[:6]:
    if path.exists() and path.suffix == '.csv':
        loaded[path.name] = pd.read_csv(path)

if 'external_molecules_87.csv' in loaded and 'structural_neighbor_pairs_60.csv' in loaded:
    counts = pd.DataFrame([
        {'item': 'external molecules', 'observed': len(loaded['external_molecules_87.csv']), 'expected': DATASET_CFG['dataset']['external_n_molecules']},
        {'item': 'external structural-neighbor pairs', 'observed': len(loaded['structural_neighbor_pairs_60.csv']), 'expected': DATASET_CFG['dataset']['external_n_pairs']},
    ])
    display(counts)
else:
    print('External molecule and pair count checks require the two primary external tables.')

## 6. Panel vocabulary check

In [ ]:
if 'panel_label_vocabulary_43.csv' in loaded:
    vocab = loaded['panel_label_vocabulary_43.csv']
    print('Observed panel vocabulary size:', len(vocab))
    print('Expected panel vocabulary size:', DATASET_CFG['dataset']['panel_n_labels'])
    display(vocab.head())
else:
    print('Panel vocabulary check requires panel_label_vocabulary_43.csv.')

## 7. Aggregated sensory rating checks

In [ ]:
if 'aggregated_panel_ratings.csv' in loaded:
    ratings = loaded['aggregated_panel_ratings.csv']
    print('Columns:', ratings.columns.tolist())
    if 'n_assessors' in ratings.columns:
        print('Unique n_assessors values:', sorted(ratings['n_assessors'].dropna().unique().tolist()))
    display(ratings.head())
    print('Reminder: this notebook is intended for aggregated and anonymized panel data only.')
else:
    print('Aggregated sensory rating checks require aggregated_panel_ratings.csv.')

## 8. Panel-derived Top-K label checks

In [ ]:
if 'panel_derived_topk_labels.csv' in loaded:
    topk = loaded['panel_derived_topk_labels.csv']
    display(topk.head())
    if {'molecule_id', 'panel_label'}.issubset(topk.columns):
        print('Molecules with at least one derived panel label:', topk['molecule_id'].nunique())
else:
    print('Top-K label checks require panel_derived_topk_labels.csv.')

## 9. Pairwise Jaccard similarity check

In [ ]:
if 'pairwise_panel_jaccard_similarity.csv' in loaded:
    pairwise = loaded['pairwise_panel_jaccard_similarity.csv']
    display(pairwise.head())
    if 'jaccard_similarity' in pairwise.columns:
        display(pairwise['jaccard_similarity'].describe().to_frame(name='jaccard_similarity'))
else:
    print('Pairwise Jaccard similarity check requires pairwise_panel_jaccard_similarity.csv.')

## 10. Graph-model prediction comparison

In [ ]:
graph_pred_path = PROJECT_ROOT / 'results/external_validation/graph_pairwise_similarity_predictions.csv'
if graph_pred_path.exists():
    graph_pred = pd.read_csv(graph_pred_path)
    display(graph_pred.head())
else:
    print('Graph-model prediction comparison requires the released prediction summary table.')

## 11. Concordance coefficient summary

In [ ]:
ccc_path = PROJECT_ROOT / 'results/external_validation/concordance_coefficients.csv'
if ccc_path.exists():
    ccc = pd.read_csv(ccc_path)
    display(ccc.head())
else:
    print('Concordance coefficient summary requires concordance_coefficients.csv.')

## 12. Embedding kNN macro-AUC summary

In [ ]:
knn_path = PROJECT_ROOT / 'results/external_validation/embedding_knn_auc.csv'
if knn_path.exists():
    knn = pd.read_csv(knn_path)
    display(knn.head())
else:
    print('Embedding kNN macro-AUC summary requires embedding_knn_auc.csv.')

## 13. Missing data checklist

In [ ]:
missing_items = file_report.loc[~file_report['exists'], 'path'].tolist()
if missing_items:
    print('Data required from user later:')
    for item in missing_items:
        print(f'- {item}')
else:
    print('No missing external validation input files detected for this notebook.')